# Resultados del pronóstico municipal 6→6, clúster 4

[← Metodología común](metodologia-pronostico-6x6.ipynb)

Este notebook presenta exclusivamente la configuración y los resultados de
`cluster4`. El entrenamiento comienza en `2014-01-01` y el detalle inicial se
centra en **Las Condes**.


> **Precaución para `cluster4`:** este clúster contiene únicamente a Las Condes. La
> variable comuna es constante, WAPE macro y micro coinciden, y los modelos complejos
> tienen mayor riesgo de sobreajuste. Se conserva el mismo protocolo para hacer la
> comparación temporal, pero el resultado debe interpretarse como un pronóstico
> individual y exploratorio.


In [9]:
# Configuración editable del notebook
from utils.forecasting_cache import ForecastCacheConfig

CLUSTER_COLUMN = "cluster4"
TRAINING_START = "2014-01-01"
PREFERRED_MUNICIPALITY = "Las Condes"

MODEL_CACHE = ForecastCacheConfig(
    directory="model_cache",
    enabled=True,
    refresh=False,
)


## Composición del clúster 4

La celda resuelve las comunas marcadas en `cluster4` y deja visible la configuración
efectiva antes de cargar los datos.


In [10]:
import pandas as pd
from IPython.display import HTML, display

from utils.data import data_loader
from utils.forecasting_cache import summarize_cache_report
from utils.forecasting_workflow import (
    build_municipality_forecast_view,
    macro_micro_gap,
    resolve_cluster_municipalities,
    run_cluster_forecasting_workflow,
)
from utils.municipal_income import build_municipalities_income_history
from utils.notebook_display import collapsible_stdout, display_collapsible
from utils.plots import (
    plot_annual_income_share_lines,
    plot_municipality_forecast_evaluation,
)

clusters = pd.read_csv("clusters.csv")
municipalities = resolve_cluster_municipalities(clusters, CLUSTER_COLUMN)
display_collapsible(
    "Ver configuración efectiva",
    pd.DataFrame(
        {
            "cluster": [CLUSTER_COLUMN],
            "comunas": [len(municipalities)],
            "inicio de entrenamiento": [pd.Timestamp(TRAINING_START)],
            "comuna para detalle": [PREFERRED_MUNICIPALITY],
            "caché habilitado": [MODEL_CACHE.enabled],
            "refrescar caché": [MODEL_CACHE.refresh],
            "directorio caché": [str(MODEL_CACHE.directory)],
        }
    )
)
municipality_label = "comuna" if len(municipalities) == 1 else "comunas"
municipality_items = "".join(
    f"<li>{municipality}</li>" for municipality in municipalities
)
display_collapsible(
    f"Ver {len(municipalities)} {municipality_label} del clúster",
    HTML(
        f'<ul class="cluster-members__list">{municipality_items}</ul>'
    )
)


,cluster,comunas,inicio de entrenamiento,comuna para detalle,caché habilitado,refrescar caché,directorio caché
0,cluster4,1,2014-01-01,Las Condes,True,False,model_cache


## Historia descriptiva del clúster 4

Se muestran trayectorias independientes para las comunas de `cluster4` antes de
presentar la evaluación predictiva.


In [11]:
with collapsible_stdout("Ver registro de carga de datos"):
    presupuesto_cluster = data_loader(municipalities=municipalities)
historial_cluster = build_municipalities_income_history(
    presupuesto_cluster,
    municipalities=municipalities,
)

for municipality in municipalities:
    municipality_history = historial_cluster.loc[
        historial_cluster["Nombre Municipio"].eq(municipality)
    ]
    figure = plot_annual_income_share_lines(municipality_history)
    year_count = int(municipality_history["Ejercicio"].nunique())
    figure.update_layout(width=max(900, year_count * 90 + 240), autosize=False)
    figure_html = figure.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config={"responsive": False, "displaylogo": False},
    )
    display(
        HTML(
            '<div style="max-width:100%; overflow-x:auto; padding-bottom:1rem;">'
            f"{figure_html}</div>"
        )
    )


## Ejecución y trazabilidad del clúster 4

El bloque ejecuta el workflow conjunto para `cluster4`. El contrato temporal, las
métricas y el comportamiento del caché están documentados en la
[metodología común](metodologia-pronostico-6x6.ipynb).


In [12]:
with collapsible_stdout("Ver registro de entrenamiento conjunto"):
    workflow = run_cluster_forecasting_workflow(
        presupuesto_cluster,
        municipalities,
        cluster_label=CLUSTER_COLUMN,
        training_start=TRAINING_START,
        progress=True,
        cache_config=MODEL_CACHE,
    )

display_collapsible("Ver configuración del workflow", workflow.configuration)
display_collapsible("Ver ventanas de entrenamiento", workflow.training_windows)
display_collapsible("Ver progresión de modelos", workflow.model_progression)
display_collapsible(
    "Ver resumen del caché",
    summarize_cache_report(workflow.cache_report),
)

cache_by_stage = (
    workflow.cache_report.groupby(
        ["stage", "artifact_type", "status"], sort=False, dropna=False
    )
    .agg(artefactos=("key", "size"), gb=("size_bytes", lambda x: x.sum() / 1024**3))
    .reset_index()
)
display_collapsible("Ver artefactos de caché por etapa", cache_by_stage)


,cluster,comunas,inicio_entrenamiento_solicitado,primer_corte_tuning,ultimo_corte_tuning,primer_corte_validacion,ultimo_corte_validacion,fin_entrenamiento_final,entrada_final,test_congelado,ventanas_entrenamiento_final,variables_mensuales,objetivos_directos
0,cluster4,1,2014-01-01,2018-06-01,2022-12-01,2023-06-01,2024-06-01,2025-06-01,julio-diciembre 2025,enero-junio 2026,127,30,30


,Nombre Municipio,primera_entrada,ultimo_objetivo,ventanas
0,Las Condes,2014-01-01,2025-06-01,127


,nivel,modelo,complejidad
0,1,Persistencia,Repite el último vector mensual; no se entrena.
1,2,Ridge global,Relación lineal regularizada; comuna y mes one...
2,3,ExtraTrees global,Ensamble no lineal; comuna y mes one-hot.
3,4,CatBoost global,Boosting no lineal; comuna y mes categóricos n...


,hits,misses,modelos_cargados,evaluaciones_cargadas,reutilizaciones_memoria,artefactos_escritos,artefactos_corruptos_aislados,gb_cargados,gb_escritos,gb_artefactos_utilizados
0,117,0,3,114,0,0,0,0.012157,0.0,0.012157


,stage,artifact_type,status,artefactos,gb
0,joint_tuning,evaluation,hit,110,0.000179
1,joint_validation,evaluation,hit,3,0.000007
2,joint_final_model,model,hit,3,0.011970
3,joint_test,evaluation,hit,1,0.000002


## Hiperparámetros elegidos para el clúster 4

La tabla identifica las configuraciones retenidas para `cluster4` antes de comparar
familias en validación.


In [13]:
tuning_table = workflow.tuning_results.copy()
tuning_table["WAPE macro (%)"] = (100 * tuning_table["wape_macro"]).round(2)
tuning_table["WAPE micro (%)"] = (100 * tuning_table["wape_micro"]).round(2)
tuning_table["MAE (MM CLP)"] = tuning_table["mae_mm_clp"].round(2)
selected_parameters = pd.DataFrame(
    [
        {
            "modelo": spec.name,
            "candidate_id": spec.candidate_id,
            "parametros": dict(spec.params),
        }
        for spec in workflow.selected_specs
    ]
)
display_collapsible(
    "Ver resultados de tuning",
    tuning_table[
        [
            "familia",
            "modelo_candidato",
            "parametros",
            "seleccionado",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
        ]
    ]
)
display_collapsible("Ver hiperparámetros seleccionados", selected_parameters)


,familia,modelo_candidato,parametros,seleccionado,WAPE macro (%),WAPE micro (%),MAE (MM CLP)
0,ridge,Ridge global [alpha=0.1],{'alpha': 0.1},True,17.07,17.07,2832.36
1,ridge,Ridge global [alpha=1],{'alpha': 1.0},False,18.48,18.48,3065.78
2,ridge,Ridge global [alpha=10],{'alpha': 10.0},False,23.58,23.58,3911.98
3,extra_trees,"ExtraTrees global [depth=none, leaf=1]","{'n_estimators': 400, 'max_depth': None, 'min_...",True,18.98,18.98,3150.15
4,extra_trees,"ExtraTrees global [depth=12, leaf=1]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,19.05,19.05,3161.28
5,extra_trees,"ExtraTrees global [depth=12, leaf=3]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,19.54,19.54,3242.42
6,extra_trees,"ExtraTrees global [depth=none, leaf=3]","{'n_estimators': 400, 'max_depth': None, 'min_...",False,19.55,19.55,3243.97
7,catboost,"CatBoost global [depth=4, iterations=500]","{'iterations': 500, 'depth': 4, 'learning_rate...",True,25.23,25.23,4186.78
8,catboost,"CatBoost global [depth=6, iterations=500]","{'iterations': 500, 'depth': 6, 'learning_rate...",False,26.68,26.68,4427.65
9,catboost,"CatBoost global [depth=4, iterations=300]","{'iterations': 300, 'depth': 4, 'learning_rate...",False,27.52,27.52,4566.16


,modelo,candidate_id,parametros
0,Ridge global,ridge_alpha_0.1,{'alpha': 0.1}
1,ExtraTrees global,extra_trees_depth_none_leaf_1,"{'n_estimators': 400, 'max_depth': None, 'min_..."
2,CatBoost global,catboost_depth_4_iterations_500,"{'iterations': 500, 'depth': 4, 'learning_rate..."


## Selección histórica del clúster 4

Este bloque muestra el ganador congelado de `cluster4` y sus métricas por grupo de
ingreso antes de abrir el test de 2026.


In [14]:
validation_ranking = workflow.validation_ranking.copy()
validation_ranking["WAPE macro (%)"] = (
    100 * validation_ranking["wape_macro"]
).round(2)
validation_ranking["WAPE micro (%)"] = (
    100 * validation_ranking["wape_micro"]
).round(2)

validation_metrics = workflow.validation_summary.copy()
validation_metrics["WAPE macro (%)"] = (
    100 * validation_metrics["wape_macro"]
).round(2)
validation_metrics["WAPE micro (%)"] = (
    100 * validation_metrics["wape_micro"]
).round(2)
validation_metrics["MAE (MM CLP)"] = validation_metrics["mae_mm_clp"].round(2)
validation_metrics["Sesgo (MM CLP)"] = (
    validation_metrics["sesgo_micro_mm_clp"].round(2)
)

display_collapsible(
    "Ver ranking de validación",
    validation_ranking[
        [
            "ranking",
            "seleccion_validacion",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas de validación por ingreso",
    validation_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
        ]
    ]
)
display_collapsible(
    "Ver modelo seleccionado por validación",
    workflow.selected_model_name,
)


,ranking,seleccion_validacion,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,Ridge global,12.00,12.00,2544.598120,-1827.405698
1,2,False,ExtraTrees global,19.57,19.57,4150.797549,-4056.499454
2,3,False,CatBoost global,29.27,29.27,6207.654035,-3982.809380
3,4,False,Persistencia,69.40,69.40,14717.465568,-4244.288350


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP)
0,CatBoost global,FCM,24.89,24.89,107.81,-60.72
1,CatBoost global,IPP,31.95,31.95,5814.06,-3398.55
2,CatBoost global,Otros ingresos,49.35,49.35,488.72,-202.57
3,CatBoost global,Total disponible,29.27,29.27,6207.65,-3982.81
4,CatBoost global,Transferencias corrientes,21.49,21.49,340.29,-321.55
5,CatBoost global,Transferencias de capital,221.88,221.88,10.38,0.57
6,ExtraTrees global,FCM,17.16,17.16,74.32,-73.85
7,ExtraTrees global,IPP,19.98,19.98,3635.22,-3522.99
8,ExtraTrees global,Otros ingresos,58.38,58.38,578.09,-149.83
9,ExtraTrees global,Total disponible,19.57,19.57,4150.80,-4056.50


## Test congelado de 2026 para el clúster 4

La comparación fuera de muestra de `cluster4` aplica la cobertura observada definida
en la [metodología común](metodologia-pronostico-6x6.ipynb).


In [15]:
test_ranking = workflow.test_ranking.copy()
test_ranking["WAPE macro (%)"] = (100 * test_ranking["wape_macro"]).round(2)
test_ranking["WAPE micro (%)"] = (100 * test_ranking["wape_micro"]).round(2)

test_metrics = workflow.test_summary.copy()
test_metrics["WAPE macro (%)"] = (100 * test_metrics["wape_macro"]).round(2)
test_metrics["WAPE micro (%)"] = (100 * test_metrics["wape_micro"]).round(2)
test_metrics["MAE (MM CLP)"] = test_metrics["mae_mm_clp"].round(2)
test_metrics["Sesgo (MM CLP)"] = test_metrics["sesgo_micro_mm_clp"].round(2)

display_collapsible(
    "Ver ranking del test 2026",
    test_ranking[
        [
            "ranking",
            "mejor_test",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas del test 2026 por ingreso",
    test_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
            "meses_evaluados",
        ]
    ]
)

if len(workflow.municipalities) == 1:
    assert macro_micro_gap(workflow.test_summary).fillna(0).le(1e-12).all()


,ranking,mejor_test,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,Ridge global,8.61,8.61,1865.090332,26.199216
1,2,False,ExtraTrees global,19.91,19.91,4312.077780,-4312.077780
2,3,False,CatBoost global,26.54,26.54,5747.682019,-2032.323967
3,4,False,Persistencia,66.68,66.68,14440.234632,-1028.631947


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP),meses_evaluados
0,CatBoost global,FCM,31.30,31.30,124.35,4.63,6.0
1,CatBoost global,IPP,31.52,31.52,5869.55,-2030.29,6.0
2,CatBoost global,Otros ingresos,42.79,42.79,357.48,253.24,6.0
3,CatBoost global,Total disponible,26.54,26.54,5747.68,-2032.32,6.0
4,CatBoost global,Transferencias corrientes,15.68,15.68,280.38,-247.18,6.0
5,CatBoost global,Transferencias de capital,119.02,119.02,16.04,-12.72,6.0
6,ExtraTrees global,FCM,16.01,16.01,63.62,-59.40,6.0
7,ExtraTrees global,IPP,22.97,22.97,4278.24,-4278.24,6.0
8,ExtraTrees global,Otros ingresos,93.83,93.83,783.90,459.30,6.0
9,ExtraTrees global,Total disponible,19.91,19.91,4312.08,-4312.08,6.0


## Detalle inicial de Las Condes

`PREFERRED_MUNICIPALITY` permite cambiar la comuna inspeccionada dentro de `cluster4`
sin alterar la selección ni el ranking ya calculados.


In [16]:
municipality_view = build_municipality_forecast_view(
    workflow,
    PREFERRED_MUNICIPALITY,
)
display_collapsible(
    f"Ver métricas de {municipality_view.municipality}",
    municipality_view.metrics,
)

forecast_figure = plot_municipality_forecast_evaluation(
    municipality_view.actual,
    municipality_view.forecasts,
    municipality_view.metrics,
    municipality=municipality_view.municipality,
)
display(
    HTML(
        forecast_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,Nombre Municipio,modelo,grupo_ingreso,mae_mm_clp,sesgo_mm_clp,suma_error_absoluto,suma_observado_absoluto,wape,meses_evaluados
0,Las Condes,CatBoost global,FCM,124.345200,4.631790,746.071199,2383.511219,0.313014,6.0
1,Las Condes,CatBoost global,IPP,5869.548577,-2030.291093,35217.291463,111733.968809,0.315189,6.0
2,Las Condes,CatBoost global,Otros ingresos,357.478954,253.241246,2144.873726,5012.878720,0.427873,6.0
3,Las Condes,CatBoost global,Total disponible,5747.682019,-2032.323967,34486.092112,129940.769506,0.265399,6.0
4,Las Condes,CatBoost global,Transferencias corrientes,280.377214,-247.182658,1682.263285,10729.532086,0.156788,6.0
5,Las Condes,CatBoost global,Transferencias de capital,16.043303,-12.723251,96.259818,80.878672,1.190176,6.0
6,Las Condes,ExtraTrees global,FCM,63.617660,-59.404981,381.705958,2383.511219,0.160144,6.0
7,Las Condes,ExtraTrees global,IPP,4278.241748,-4278.241748,25669.450488,111733.968809,0.229737,6.0
8,Las Condes,ExtraTrees global,Otros ingresos,783.901216,459.297852,4703.407293,5012.878720,0.938265,6.0
9,Las Condes,ExtraTrees global,Total disponible,4312.077780,-4312.077780,25872.466683,129940.769506,0.199110,6.0
